# 지마켓 실습 (상품명|상품 가격)

In [1]:
# 브라우저 역할의 웹드라이버 로딩
from selenium import webdriver as wb

# 구분자
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# 진행상황을 progress bar로 시각화해주는 라이브러리
from tqdm import tqdm

# 지연시간을 줌
import time
    # 지연시간 없이 크롤링 할 경우 봇으로 인식되어 접속 막힘
    # 아무리 인터넷이 빠르고 좋아도 여전히 페이지는 데이터를 로딩하는 시간이 필요하다
    # 이 부분이 매치가 안될경우, 데이터가 뽑히지 않을 수 있다.

In [40]:
# 사용자가 크롬창을 열듯, 우리도 컴퓨터에게 명령하자
driver = wb.Chrome()
driver.get('https://corners.gmarket.co.kr/bestsellers')

# 웹페이지가 축소 될 경우, HTML 구조가 변경될 수 있다.
    # 최대창으로 진행을 권장
driver.maximize_window()

### 이미지 수집

In [10]:
body = driver.find_element(By.TAG_NAME, 'body')
body.send_keys(Keys.END)

In [41]:
# body로 지정해서 스크롤을 쭉 내리면, 타이틀 수집이 잘 안되었다.
    # body변수 선언을 고려하되, 우선은 img를 클릭해서 들어가보자.
    
imgs = driver.find_elements(By.CLASS_NAME,'image__lazy.image__load')

# 이미지 클릭
imgs[0].click()

# 수집한 이미지의 갯수를 확인
len(imgs)

20

# 상품 명을 수집

In [42]:
# 테스트 진행
title_test = driver.find_elements(By.CLASS_NAME,'itemtit')

# for loop으로 제목이 잘 뽑히는지 테스트를 한번 해보자
for i in title_test:
    print(i.text)

하남쭈꾸미쭈꾸미볶음 500g 3팩


In [30]:
# 엇 find_elements로 추출했더니,
    # 추출된 title이 하나네?
    # 그러면 find_element로 해도 괜찮겠다.
title = driver.find_element(By.CLASS_NAME,'itemtit')
title.text

'하남쭈꾸미쭈꾸미볶음 500g 3팩'

### 상품 가격을 수집

In [43]:
# 요소들이 몇개나 뽑히는지 테스트를 한번 해보자
price_test = driver.find_elements(By.CLASS_NAME,'price_real')

# 요소'들'이기에 for loop을 사용
for i in price_test:
    print(i.text)

33,900원
28,820원


In [44]:
# 엇 이번에도 find_element로 출력해도 무방하겠다.
price = driver.find_element(By.CLASS_NAME,'price_real')
price.text

'33,900원'

In [45]:
# 품목 페이지로 돌아오자!
driver.back()

### 이제 대망의 for loop으로 추출을 해보자 

In [46]:
# 먼저 빈 그릇, (빈 리스트) 2개 생성

title_list = []
price_list = []

# 품목 10개를 가져오자
for i in tqdm(range(11)):
    imgs = driver.find_elements(By.CLASS_NAME,'image__lazy.image__load')
    imgs[i].click()
    
    time.sleep(2)
    # 지연시간을 2초 정도 줌, 
        # 데이터 로딩속도 고려했을 때, 2초에서 3초정도가 적당
        
    title = driver.find_element(By.CLASS_NAME,'itemtit')
    title_list.append(title.text)
    
    price = driver.find_element(By.CLASS_NAME,'price_real')
    price_list.append(price.text)
    
    driver.back()
    
    time.sleep(2)
    
    

100%|██████████████████████████████████████████████████████████████████████████████████| 11/11 [02:06<00:00, 11.51s/it]


In [59]:
len(title_list),len(price_list)

(11, 11)

# DataFrame 

In [60]:
dic = {"상품명":title_list,"가격":price_list}

In [64]:
import pandas as pd
GMARKET_df = pd.DataFrame(dic)
GMARKET_df

,상품명,가격
0,하남쭈꾸미쭈꾸미볶음 500g 3팩,"33,900원"
1,국내산 재료로 담근 총각김치 알타리 3kg /HACCP인증/당일제조,"17,500원"
2,(쿠폰적용필수)(첫구매/웰컴백 고객 대상) (CGV) 가디언즈 오브 갤럭시 전용 예매권,"15,000원"
3,초특가 비비고 한섬만두 320g8팩,"25,000원"
4,탱글탱글 생 새우살 200g x 3봉 / 쿠폰가 12400 (5/3하루특가),"15,500원"
5,항공직송 남독마이 햇 생망고 4kg내외(8-12과),"40,890원"
6,펩시콜라 제로 210ml x 30캔/탄산음료/제로콜라/음료수/펩시제로,"17,900원"
7,스윗마토 스테비아 방울 토마토 /망고향 달콤 토마토 500g X 4팩,"21,900원"
8,(스마일클럽)일반예매권2D-주중/주말,"8,900원"
9,목우촌주부9단 로스구이 500g+130g x 3개,"25,700원"


# 파일로 내보내기

In [66]:
# csv로 내보내기
GMARKET_df.to_csv('C:/Users/user07/webcrawling/Gmarket_items.csv',encoding = 'euc-kr')

In [68]:
# excal로 내보내기
GMARKET_df.to_excel('C:/Users/user07/webcrawling/Gmarket_items.xlsx',encoding = 'euc-kr')

C:\Users\user07\anaconda3\lib\site-packages\pandas\util\_decorators.py:211: FutureWarning: the 'encoding' keyword is deprecated and will be removed in a future version. Please take steps to stop the use of 'encoding'
  return func(*args, **kwargs)
